# 🧠 Notebook 04 — Temporal Fusion Transformer (TFT)
## Train a multi-series, multi-horizon global model that predicts 12 months ahead (ordered_qty) using both ERP and external data, providing quantile forecasts (P10/P50/P90) and variable importances.

### 1️⃣ Imports & Setup

In [1]:
# Install packages if not already installed
# !pip install pytorch-lightning pytorch-forecasting pandas numpy

import os
import warnings
import numpy as np
import pandas as pd
import torch

# PyTorch Forecasting
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss

# PyTorch Lightning
try:
    import lightning.pytorch as pl
    from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor, ModelCheckpoint
except Exception:
    import pytorch_lightning as pl
    from pytorch_lightning.callbacks import EarlyStopping, LearningRateMonitor, ModelCheckpoint

# For reproducibility
pl.seed_everything(42, workers=True)


Seed set to 42


42

### 2️⃣ Load the processed data

In [ ]:
# Load CSV
data_path = "data/training/training_data_final.csv"
df = pd.read_csv(data_path, low_memory=False)

# Ensure minimal columns exist
required_cols = {"customer_id", "product_id", "product_group", "Date", "ordered_qty"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Convert Date, create time_idx, handle NaNs
def month_features_from_date(dt: pd.Series) -> pd.DataFrame:
    out = pd.DataFrame(index=dt.index)
    out["Year"] = dt.dt.year.astype(int)
    out["Month"] = dt.dt.month.astype(int)
    out["Quarter"] = dt.dt.quarter.astype(int)
    out["month_sin"] = np.sin(2 * np.pi * out["Month"] / 12)
    out["month_cos"] = np.cos(2 * np.pi * out["Month"] / 12)
    return out

def prepare_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    if df["Date"].isna().any():
        raise ValueError("Date column contains unparsable values.")
    
    min_date = df["Date"].min()
    df["time_idx"] = (df["Date"].dt.year - min_date.year) * 12 + (df["Date"].dt.month - min_date.month)
    df["time_idx"] = df["time_idx"].astype(int)
    
    tf = month_features_from_date(df["Date"])
    for c in tf.columns:
        df[c] = tf[c]

    df["customer_id"] = df["customer_id"].astype(str)
    df["product_id"] = df["product_id"].astype(str)
    df["product_group"] = df["product_group"].astype(str)
    df["series_id"] = df["customer_id"] + "_" + df["product_id"]

    num_cols = df.select_dtypes(include=["number"]).columns
    df[num_cols] = df[num_cols].replace([np.inf, -np.inf], np.nan)
    df[num_cols] = df[num_cols].fillna(0.0)
    
    return df

df = prepare_dataframe(df)
df.head()


,customer_id,product_id,customer_name,product_group,product_name,Date,ordered_qty,order_amount,unit_price,ordered_qty_observed,order_amount_observed,unit_price_observed,unit_price_missing,Food_Price_Index,Electricity_Price,ECB_Inflation,ECB_Interest_Rate,FEDFUNDS_Value,Purchase_Index,Total_New_Orders_Value,Steel_Price,Food_Price_Index_is_real,Electricity_Price_is_real,ECB_Inflation_is_real,ECB_Interest_Rate_is_real,FEDFUNDS_Value_is_real,Purchase_Index_is_real,Total_New_Orders_Value_is_real,Steel_Price_is_real,Food_Price_Index_missing,Food_Price_Index_lag1,Food_Price_Index_lag2,Food_Price_Index_lag3,Food_Price_Index_rollmean3,Food_Price_Index_rollstd3,Food_Price_Index_rollmean6,Food_Price_Index_rollstd6,Electricity_Price_missing,Electricity_Price_lag1,Electricity_Price_lag2,...,Steel_Price_rollmean6,Steel_Price_rollstd6,Year,Month,Quarter,month_sin,month_cos,ordered_qty_lag1,ordered_qty_lag2,ordered_qty_lag3,ordered_qty_lag6,ordered_qty_lag12,ordered_qty_rollmean3,ordered_qty_rollstd3,ordered_qty_rollmean6,ordered_qty_rollstd6,ordered_qty_rollmean12,ordered_qty_rollstd12,ordered_qty_yoy_change,ordered_qty_lag12_is_zero,order_amount_lag1,order_amount_lag2,order_amount_lag3,order_amount_lag6,order_amount_lag12,order_amount_rollmean3,order_amount_rollstd3,order_amount_rollmean6,order_amount_rollstd6,order_amount_rollmean12,order_amount_rollstd12,order_amount_yoy_change,order_amount_lag12_is_zero,qty_change_1_3,qty_change_3_6,months_since_first,y_qty,y_revenue,time_idx,series_id
0,7,9103373-A,TUME-AGRI,5002,0HH04503502003000505 35/20-300 A505,2015-01-01,0.0,0.0,0.0,0,0,0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,2015,1,1,0.500000,8.660254e-01,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0,0.0,0.0,0,6.0,960.0,0,7_9103373-A
1,7,9103373-A,TUME-AGRI,5002,0HH04503502003000505 35/20-300 A505,2015-02-01,6.0,960.0,160.0,1,1,1,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,2015,2,1,0.866025,5.000000e-01,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0,0.0,0.0,1,0.0,0.0,1,7_9103373-A
2,7,9103373-A,TUME-AGRI,5002,0HH04503502003000505 35/20-300 A505,2015-03-01,0.0,0.0,160.0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,2015,3,1,1.000000,6.123234e-17,6.0,0.0,0.0,0.0,0.0,3.0,4.242641,3.0,4.242641,3.0,4.242641,0.0,0,960.0,0.0,0.0,0.0,0.0,480.0,678.822510,480.0,678.822510,480.0,678.822510,0.0,0,0.0,0.0,2,0.0,0.0,2,7_9103373-A
3,7,9103373-A,TUME-AGRI,5002,0HH04503502003000505 35/20-300 A505,2015-04-01,0.0,0.0,160.0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,2015,4,2,0.866025,-5.000000e-01,0.0,6.0,0.0,0.0,0.0,2.0,3.464102,2.0,3.464102,2.0,3.464102,0.0,0,0.0,960.0,0.0,0.0,0.0,320.0,554.256258,320.0,554.256258,320.0,554.256258,0.0,0,0.0,0.0,3,0.0,0.0,3,7_9103373-A
4,7,9103373-A,TUME-AGRI,5002,0HH04503502003000505 35/20-300 A505,2015-05-01,0.0,0.0,160.0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,2015,5,2,0.500000,-8.660254e-01,0.0,0.0,6.0,0.0,0.0,2.0,3.464102,1.5,3.000000,1.5,3.000000,0.0,0,0.0,0.0,960.0,0.0,0.0,320.0,554.256258,240.0,480.000000,240.0,480.000000,0.0,0,0.0,0.0,4,0.0,0.0,4,7_9103373-A


In [3]:
# Parameters
target = "ordered_qty"
max_encoder_length = 60
max_prediction_length = 12

max_time = df["time_idx"].max()
test_cutoff = max_time - max_prediction_length
val_cutoff = max_time - 2 * max_prediction_length
train_cutoff = max_time - 3 * max_prediction_length

# Time-varying and static features
known_reals = ["time_idx", "Year", "Month", "Quarter", "month_sin", "month_cos"]
static_categoricals = ["customer_id", "product_id", "product_group"]
time_varying_unknown_reals = [target, "order_amount", "unit_price"]

# Training dataset
train_df = df[df.time_idx <= test_cutoff]
training = TimeSeriesDataSet(
    train_df,
    time_idx="time_idx",
    target=target,
    group_ids=["series_id"],
    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,
    static_categoricals=static_categoricals + ["series_id"],
    time_varying_known_reals=known_reals,
    time_varying_unknown_reals=time_varying_unknown_reals,
    target_normalizer=GroupNormalizer(groups=["series_id"], transformation="softplus"),
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
    allow_missing_timesteps=False,
)

# Validation dataset
validation = TimeSeriesDataSet.from_dataset(
    training, train_df, predict=True, stop_randomization=True
)

# Test dataset (last 12 months)
test = TimeSeriesDataSet.from_dataset(
    training, df, predict=True, stop_randomization=True
)

print("Training/validation/test datasets ready.")


Training/validation/test datasets ready.


In [4]:
batch_size = 256
num_workers = 0

train_loader = training.to_dataloader(train=True, batch_size=batch_size, num_workers=num_workers)
val_loader = validation.to_dataloader(train=False, batch_size=batch_size, num_workers=num_workers)
test_loader = test.to_dataloader(train=False, batch_size=batch_size, num_workers=num_workers)


In [5]:
loss = QuantileLoss()
model = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=3e-3,
    hidden_size=32,
    attention_head_size=4,
    dropout=0.1,
    lstm_layers=1,
    loss=loss,
    log_interval=10,
    reduce_on_plateau_patience=4,
)
model


D:\anaconda3\envs\tensor\lib\site-packages\lightning\pytorch\utilities\parsing.py:210: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
D:\anaconda3\envs\tensor\lib\site-packages\lightning\pytorch\utilities\parsing.py:210: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.


TemporalFusionTransformer(
  	"attention_head_size":               4
  	"categorical_groups":                {}
  	"causal_attention":                  True
  	"dataset_parameters":                {'time_idx': 'time_idx', 'target': 'ordered_qty', 'group_ids': ['series_id'], 'weight': None, 'max_encoder_length': 60, 'min_encoder_length': 60, 'min_prediction_idx': 0, 'min_prediction_length': 12, 'max_prediction_length': 12, 'static_categoricals': ['customer_id', 'product_id', 'product_group', 'series_id'], 'static_reals': None, 'time_varying_known_categoricals': None, 'time_varying_known_reals': ['time_idx', 'Year', 'Month', 'Quarter', 'month_sin', 'month_cos'], 'time_varying_unknown_categoricals': None, 'time_varying_unknown_reals': ['ordered_qty', 'order_amount', 'unit_price'], 'variable_groups': None, 'constant_fill_strategy': None, 'allow_missing_timesteps': False, 'lags': None, 'add_relative_time_idx': True, 'add_target_scales': True, 'add_encoder_length': True, 'target_normalizer':

In [7]:
out_dir = "outputs"
os.makedirs(out_dir, exist_ok=True)
ckpt_dir = os.path.join(out_dir, "checkpoints")
os.makedirs(ckpt_dir, exist_ok=True)

checkpoint_cb = ModelCheckpoint(
    dirpath=ckpt_dir,
    filename="tft-{epoch:02d}-{val_loss:.4f}",
    monitor="val_loss",
    save_top_k=1,
    mode="min",
)

callbacks = [
    EarlyStopping(monitor="val_loss", patience=8, mode="min"),
    LearningRateMonitor(logging_interval="epoch"),
    checkpoint_cb,
]

trainer = pl.Trainer(
    max_epochs=20,
    accelerator="auto",
    devices=1,
    gradient_clip_val=0.1,
    callbacks=callbacks,
    enable_checkpointing=True,
    log_every_n_steps=50,
)


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [8]:
trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)



   | Name                               | Type                            | Params | Mode 
------------------------------------------------------------------------------------------------
0  | loss                               | QuantileLoss                    | 0      | train
1  | logging_metrics                    | ModuleList                      | 0      | train
2  | input_embeddings                   | MultiEmbedding                  | 173 K  | train
3  | prescalers                         | ModuleDict                      | 208    | train
4  | static_variable_selection          | VariableSelectionNetwork        | 4.2 K  | train
5  | encoder_variable_selection         | VariableSelectionNetwork        | 10.5 K | train
6  | decoder_variable_selection         | VariableSelectionNetwork        | 7.1 K  | train
7  | static_context_variable_selection  | GatedResidualNetwork            | 4.3 K  | train
8  | static_context_initial_hidden_lstm | GatedResidualNetwork            | 4.3 K  

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

D:\anaconda3\envs\tensor\lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
D:\anaconda3\envs\tensor\lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=20` reached.


In [9]:
# Validate
trainer.validate(model, dataloaders=val_loader)

# Test
trainer.test(model, dataloaders=test_loader)


D:\anaconda3\envs\tensor\lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Validation: |          | 0/? [00:00<?, ?it/s]

┌───────────────────────────┬───────────────────────────┐
│      Validate metric      │       DataLoader 0        │
├───────────────────────────┼───────────────────────────┤
│          val_MAE          │    0.42616647481918335    │
│         val_MAPE          │         8485927.0         │
│         val_RMSE          │     3.379701852798462     │
│         val_SMAPE         │    0.12155069410800934    │
│         val_loss          │    0.26902851462364197    │
└───────────────────────────┴───────────────────────────┘


D:\anaconda3\envs\tensor\lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:433: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Testing: |          | 0/? [00:00<?, ?it/s]

┌───────────────────────────┬───────────────────────────┐
│        Test metric        │       DataLoader 0        │
├───────────────────────────┼───────────────────────────┤
│         test_MAE          │    0.7511764764785767     │
│         test_MAPE         │        28542266.0         │
│         test_RMSE         │     6.172914981842041     │
│        test_SMAPE         │    0.13370411098003387    │
│         test_loss         │    0.5713260769844055     │
└───────────────────────────┴───────────────────────────┘


[{'test_loss': 0.5713260769844055,
  'test_SMAPE': 0.13370411098003387,
  'test_MAE': 0.7511764764785767,
  'test_RMSE': 6.172914981842041,
  'test_MAPE': 28542266.0}]

In [14]:
def make_future_dataframe(df, horizon=12):
    df = df.copy()
    df = df.sort_values(["series_id", "time_idx"])
    last = df.groupby("series_id").tail(1).copy()

    future_rows = []
    for step in range(1, horizon + 1):
        tmp = last.copy()
        tmp["time_idx"] = tmp["time_idx"] + step
        tmp["Date"] = (tmp["Date"] + pd.offsets.MonthBegin(step)).dt.to_period("M").dt.to_timestamp(how="start")
        
        # Add month features
        tf = month_features_from_date(tmp["Date"])
        for c in tf.columns:
            tmp[c] = tf[c]

        # Fill numeric targets with 0 (or a constant) for future rows
        tmp["ordered_qty"] = 0
        tmp["order_amount"] = 0
        
        # Optional: flags for observed
        for flag in ["ordered_qty_observed", "order_amount_observed", "unit_price_observed"]:
            if flag in tmp.columns:
                tmp[flag] = 0

        future_rows.append(tmp)

    future = pd.concat(future_rows, ignore_index=True)
    future_full = pd.concat([df, future], ignore_index=True, sort=False)
    
    return future_full


In [16]:
# Create future dataframe (next 12 months)
future_full = make_future_dataframe(df, horizon=12)

# Build prediction dataset
future_dataset = TimeSeriesDataSet.from_dataset(
    training,
    future_full,
    predict=True,
    stop_randomization=True
)

print("Future dataset ready:", len(future_dataset))


Future dataset ready: 2781


In [17]:
import torch
from pandas.core.internals.managers import BlockManager
from pytorch_forecasting import TemporalFusionTransformer

best_path = checkpoint_cb.best_model_path

if best_path:
    # Allow pandas objects used inside the checkpoint
    torch.serialization.add_safe_globals([BlockManager])

    best_model = TemporalFusionTransformer.load_from_checkpoint(best_path)
else:
    best_model = model



pred = best_model.predict(future_dataset, mode="quantiles", return_index=True, batch_size=batch_size)
preds = pred.output
idx = pred.index

# Map predictions to dates
time_to_date = future_full[["series_id", "time_idx", "Date"]].drop_duplicates()
quantiles = [0.02, 0.1, 0.25, 0.5, 0.75, 0.9, 0.98]

rows = []
n, h, q = preds.shape
for step in range(h):
    tmp = idx.copy()
    tmp["time_idx"] = tmp["time_idx"] + step
    tmp["h"] = step + 1
    tmp = tmp.merge(time_to_date, on=["series_id", "time_idx"], how="left")
    for qi, qq in enumerate(quantiles):
        tmp[f"q{int(qq*100):02d}"] = preds[:, step, qi]
    rows.append(tmp[["series_id", "Date", "time_idx", "h"] + [f"q{int(qq*100):02d}" for qq in quantiles]])

forecast_df = pd.concat(rows, ignore_index=True)
forecast_df.to_csv("outputs/tft_forecast_12m.csv", index=False)
forecast_df.head()


D:\anaconda3\envs\tensor\lib\site-packages\lightning\pytorch\utilities\parsing.py:210: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
D:\anaconda3\envs\tensor\lib\site-packages\lightning\pytorch\utilities\parsing.py:210: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
D:\anaconda3\envs\tensor\lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:433: The 'predict_dataloader' does not h

,series_id,Date,time_idx,h,q02,q10,q25,q50,q75,q90,q98
0,11682_08005001500005,2024-12-01,119,1,2.193143e-15,6.294600e-13,7.552793e-11,3.223232e-09,8.749149e-08,2.659782e-06,2.791317e-04
1,11682_08005001500010,2024-12-01,119,1,3.895801e-19,4.590138e-16,1.380748e-13,1.304701e-11,1.271726e-09,9.365525e-08,5.707225e-05
2,11682_08005004480010,2024-12-01,119,1,4.934918e-23,1.026011e-20,2.362293e-18,3.282965e-16,1.134999e-13,3.052105e-11,4.391034e-07
3,11682_10005001000005,2024-12-01,119,1,1.043422e-26,2.336116e-26,2.587208e-25,2.150000e-24,2.407556e-23,7.481950e-22,1.442147e-18
4,11682_10006001670005,2024-12-01,119,1,3.371835e-24,1.977639e-23,2.396251e-22,1.561908e-21,6.592230e-21,5.380671e-20,7.202750e-18
